In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)


Python: 3.14.6
pandas: 3.0.6
NumPy: 2.5.3
scikit-learn: 1.9.1


In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
dataset_path = Path("course_lead_scoring_2026.csv")
source = dataset_path if dataset_path.exists() else url
df_original = pd.read_csv(source)

print("Fonte oficial:", url)
print("Formato:", df_original.shape)
print(df_original.head().to_string(index=False))


Fonte oficial: https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Formato: (5000, 9)
   lead_source      industry employment_status      location  annual_income  number_of_courses_viewed  interaction_count  lead_score  converted
organic_search    technology          employed        europe        88160.0                         3                  4        0.64          1
  social_media    technology          employed south_america        72688.0                         3                  7        0.72          1
      referral        retail          employed        europe        44697.0                         3                  4        0.58          1
      paid_ads manufacturing           student           NaN            NaN                         3                  6        0.57          0
      referral manufacturing           student north_america        24062.0                         4                  5

In [3]:
target = "converted"
features = df_original.columns.drop(target).tolist()
categorical = df_original[features].select_dtypes(exclude="number").columns.tolist()
numerical = df_original[features].select_dtypes(include="number").columns.tolist()

print("Categóricas:", categorical)
print("Numéricas:", numerical)
print(df_original.isna().sum())
print("Valores do alvo:", sorted(df_original[target].unique()))
assert df_original[target].notna().all()
assert set(df_original[target].unique()) == {0, 1}


Categóricas: ['lead_source', 'industry', 'employment_status', 'location']
Numéricas: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64
Valores do alvo: [np.int64(0), np.int64(1)]


In [4]:
df = df_original.copy()
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

print(df.isna().sum())
assert df[features].notna().all().all()


lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


In [5]:
industry_counts = df["industry"].value_counts()
q1_answer = industry_counts.index[0]

print(industry_counts)
print("Resposta Q1:", q1_answer)


industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64
Resposta Q1: technology


In [6]:
correlation = df[numerical].corr()
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

pair_correlations = pd.DataFrame([
    {"feature_1": a, "feature_2": b, "correlation": correlation.loc[a, b]}
    for a, b in pairs
]).sort_values("correlation", ascending=False)

best_pair = pair_correlations.iloc[0]
q2_answer = f"{best_pair['feature_1']} and {best_pair['feature_2']}"

print(correlation.to_string())
print(pair_correlations.to_string(index=False))
print("Resposta Q2:", q2_answer)


                          annual_income  number_of_courses_viewed  interaction_count  lead_score
annual_income                  1.000000                  0.161300           0.122842    0.229496
number_of_courses_viewed       0.161300                  1.000000           0.721609    0.757204
interaction_count              0.122842                  0.721609           1.000000    0.915746
lead_score                     0.229496                  0.757204           0.915746    1.000000
               feature_1         feature_2  correlation
       interaction_count        lead_score     0.915746
number_of_courses_viewed        lead_score     0.757204
number_of_courses_viewed interaction_count     0.721609
           annual_income interaction_count     0.122842
Resposta Q2: interaction_count and lead_score


In [7]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_full_train = df_full_train[target].to_numpy()
y_train = df_train[target].to_numpy()
y_val = df_val[target].to_numpy()
y_test = df_test[target].to_numpy()

df_full_train = df_full_train.drop(columns=target).copy()
df_train = df_train.drop(columns=target).copy()
df_val = df_val.drop(columns=target).copy()
df_test = df_test.drop(columns=target).copy()

print("Treino:", df_train.shape, "Validação:", df_val.shape, "Teste:", df_test.shape)
assert all(target not in frame.columns for frame in [df_full_train, df_train, df_val, df_test])
assert set(df_train.index).isdisjoint(df_val.index)
assert set(df_train.index).isdisjoint(df_test.index)
assert set(df_val.index).isdisjoint(df_test.index)


Treino: (3000, 8) Validação: (1000, 8) Teste: (1000, 8)


In [8]:
mi_rows = []
for column in categorical:
    score = float(mutual_info_score(df_train[column], y_train))
    mi_rows.append({
        "feature": column,
        "mutual_information": score,
        "rounded_2": round(score, 2),
    })

mi_results = pd.DataFrame(mi_rows).sort_values(
    ["rounded_2", "mutual_information"], ascending=False
)
q3_answer = mi_results.iloc[0]["feature"]

print(mi_results.to_string(index=False))
print("Resposta Q3:", q3_answer)


          feature  mutual_information  rounded_2
      lead_source            0.034346       0.03
employment_status            0.018966       0.02
         industry            0.001995       0.00
         location            0.001043       0.00
Resposta Q3: lead_source


In [9]:
train_dicts = df_train[features].to_dict(orient="records")
val_dicts = df_val[features].to_dict(orient="records")

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

print("X_train:", X_train.shape, "X_val:", X_val.shape)
print(dv.get_feature_names_out())


X_train: (3000, 28) X_val: (1000, 28)
['annual_income' 'employment_status=NA' 'employment_status=employed'
 'employment_status=self_employed' 'employment_status=student'
 'employment_status=unemployed' 'industry=NA' 'industry=education'
 'industry=finance' 'industry=healthcare' 'industry=manufacturing'
 'industry=retail' 'industry=technology' 'interaction_count' 'lead_score'
 'lead_source=NA' 'lead_source=events' 'lead_source=organic_search'
 'lead_source=paid_ads' 'lead_source=referral' 'lead_source=social_media'
 'location=NA' 'location=africa' 'location=asia' 'location=europe'
 'location=north_america' 'location=south_america'
 'number_of_courses_viewed']


In [10]:
model = LogisticRegression(
    solver="liblinear", C=1.0, max_iter=1000, random_state=42
)
model.fit(X_train, y_train)

prob_val = model.predict_proba(X_val)[:, 1]
pred_val = (prob_val >= 0.50).astype(int)
original_accuracy = float(accuracy_score(y_val, pred_val))
q4_answer = round(original_accuracy, 2)

print("Acurácia sem arredondar:", original_accuracy)
print("Acertos:", int((y_val == pred_val).sum()), "de", len(y_val))
print("Resposta Q4:", q4_answer)


Acurácia sem arredondar: 0.645
Acertos: 645 de 1000
Resposta Q4: 0.65


In [11]:
def evaluate_features(selected_features, C=1.0):
    train_records = df_train[selected_features].to_dict(orient="records")
    val_records = df_val[selected_features].to_dict(orient="records")
    vectorizer = DictVectorizer(sparse=False)
    X_train_current = vectorizer.fit_transform(train_records)
    X_val_current = vectorizer.transform(val_records)
    current_model = LogisticRegression(
        solver="liblinear", C=C, max_iter=1000, random_state=42
    )
    current_model.fit(X_train_current, y_train)
    probabilities = current_model.predict_proba(X_val_current)[:, 1]
    predictions = (probabilities >= 0.50).astype(int)
    return float(accuracy_score(y_val, predictions))


In [12]:
elimination_rows = []
for removed_feature in features:
    selected_features = [feature for feature in features if feature != removed_feature]
    accuracy_without = evaluate_features(selected_features)
    elimination_rows.append({
        "removed_feature": removed_feature,
        "accuracy_without": accuracy_without,
        "difference": original_accuracy - accuracy_without,
    })

elimination_results = pd.DataFrame(elimination_rows)
q5_candidates = ["lead_source", "number_of_courses_viewed", "interaction_count"]
candidate_results = elimination_results[
    elimination_results["removed_feature"].isin(q5_candidates)
].sort_values("difference")
q5_answer = candidate_results.iloc[0]["removed_feature"]

print(elimination_results.to_string(index=False))
print("Alternativas da Q5:")
print(candidate_results.to_string(index=False))
print("Resposta Q5:", q5_answer)


         removed_feature  accuracy_without  difference
             lead_source             0.642       0.003
                industry             0.645       0.000
       employment_status             0.645       0.000
                location             0.645       0.000
           annual_income             0.724      -0.079
number_of_courses_viewed             0.643       0.002
       interaction_count             0.601       0.044
              lead_score             0.644       0.001
Alternativas da Q5:
         removed_feature  accuracy_without  difference
number_of_courses_viewed             0.643       0.002
             lead_source             0.642       0.003
       interaction_count             0.601       0.044
Resposta Q5: number_of_courses_viewed


In [13]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]
regularization_rows = []
for C in C_values:
    accuracy = evaluate_features(features, C=C)
    regularization_rows.append({
        "C": C,
        "accuracy": accuracy,
        "rounded_3": round(accuracy, 3),
    })

regularization_results = pd.DataFrame(regularization_rows)
best_regularized = regularization_results.sort_values(
    ["rounded_3", "C"], ascending=[False, True]
).iloc[0]
q6_answer = float(best_regularized["C"])

print(regularization_results.to_string(index=False))
print("Resposta Q6:", q6_answer)


       C  accuracy  rounded_3
0.000001     0.598      0.598
0.000010     0.598      0.598
0.000100     0.613      0.613
0.001000     0.645      0.645
Resposta Q6: 0.001


In [14]:
answers = {
    "Q1": q1_answer,
    "Q2": q2_answer,
    "Q3": q3_answer,
    "Q4": q4_answer,
    "Q5": q5_answer,
    "Q6": q6_answer,
}

for question, answer in answers.items():
    print(f"{question}: {answer}")


Q1: technology
Q2: interaction_count and lead_score
Q3: lead_source
Q4: 0.65
Q5: number_of_courses_viewed
Q6: 0.001
